In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("OLLAMA_API_KEY"),
    base_url="http://localhost:11434/v1"
)

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally, you can follow these steps:\n\n1. **Install Ollama**:\n   - Download and install Ollama for your operating system from [https://ollama.com/download](https://ollama.com/download).\n   - For macOS: Use the `.pkg` installer.\n   - For Windows: Use the `.msi` installer.\n   - For Linux: Run the following command in the terminal:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start the Ollama Local Server**:\n   - Once installed, open a terminal and run the following command to start the Ollama local server with the LLaMA 3 model:\n     ```bash\n     ollama run llama3\n     ```\n   - This command will download the LLaMA 3 model (~4GB) and start the model locally.\n\n3. **Test the Local Server**:\n   - To test if the Ollama local server is running correctly, run:\n     ```bash\n     curl http://localhost:11434\n     ```\n   - You should receive a JSON response similar to:\n     ```json\n     {"models": [...]}\n     ```\n\n4. **Inst

In [5]:
assistant.rag("How do I run Olama locally?")

'To run Olama locally, you need to set up a local development environment for Python and install all necessary dependencies. The setup includes:\n\n1. **Python**: Ensure you have Python installed on your system.\n2. **uv**: A tool for running asynchronous applications.\n3. **Jupyter Notebook or JupyterLab**: For interactive development and testing.\n4. **Docker**: For containerization if needed.\n\nHere are the detailed steps to set up a local environment:\n\n### Step 1: Install Python\n\nDownload and install Python from the official website. Make sure to choose the version compatible with your operating system (Windows, macOS, or Linux).\n\n### Step 2: Install Dependencies\n\nOpen a terminal or command prompt and run the following commands to install necessary Python packages:\n\n```bash\n# Install uv using pip\npip install uvicorn\n\n# Install Jupyter Notebook\npip install jupyter notebook\n\n# Optionally, if you want to use Docker\ndocker-compose up -d\n```\n\n### Step 3: Set Up Env

In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="qwen2.5-coder-3b",
    input=messages,
)

response.output_text

'Of course! Joining the course is straightforward. Here are the steps you can follow:\n\n1. **Sign Up**: If you haven\'t already registered on the platform where the course is offered, do so.\n2. **Access the Course Platform**: Navigate to the course page or section where it is listed.\n3. **Check Eligibility**: Ensure that your credentials meet the requirements for access.\n4. **Enroll in the Course**: Click on the "Join" or "Register" button to start your enrollment.\n\nIf you encounter any specific questions during this process, feel free to ask!'

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [13]:
response = openai_client.responses.create(
    model="qwen2.5-coder-3b",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseOutputMessage(id='msg_435518', content=[ResponseOutputText(annotations=[], text='```xml\n{\n    "name": "search",\n    "arguments": {\n        "query": "Can I join this course?"\n    }\n}\n```', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]

In [17]:
import json
import re

# 1. Get the first content item from the message
content_item = response.output[0].content[0]

# 2. Extract the raw text string
raw_text = content_item.text

# 3. Clean up the ```xml wrapper using regex or string stripping
# This extracts everything between the first '{' and the last '}' to get clean JSON
json_match = re.search(r'\{.*\}', raw_text, re.DOTALL)

if json_match:
    json_string = json_match.group(0)
    
    # 4. Parse the clean JSON string
    tool_call_data = json.loads(json_string)
    
    # 5. Extract the arguments and run your function
    args = tool_call_data.get("arguments", {})
    
    results = search(**args)
    result_json = json.dumps(results, indent=2)
    print(result_json)
else:
    print("Could not find a valid JSON object in the response text.")

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "85384a18e5",
    "course": "llm-zoomcamp",
    "section": "Module 1: RAG"

In [19]:
# 1. Add the assistant's text response to history
messages.extend(response.output)

# 2. Feed the search result back as a regular user message
messages.append({
    "type": "message",
    "role": "user",
    "content": f"Here are the search results for the tool call you requested:\n{result_json}"
})

In [20]:
response = openai_client.responses.create(
    model="qwen2.5-coder-3b",
    input=messages,
    tools=[search_tool],
)

response.output_text

'```xml\n{\n    "name": "search",\n    "arguments": {\n        "query": "Can I join this course?"\n    }\n}\n```'

In [21]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(964, 32)

In [23]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

## unfortenatly i use local qwen during this course so its free on my side, despite electricity and time 

Total cost: $ 0.0001176


#

In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [26]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [28]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="qwen2.5-coder-3b",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

ASSISTANT:
```json
{"name": "search", "arguments": {"query": "can i join the course"}}
```


In [29]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="qwen2.5-coder-3b",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
```json
{"name": "search", "arguments": {"query": "can i join the course"}}
```


In [31]:
def agent_loop(instructions, question, model="qwen2.5-coder-3b") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [32]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
ASSISTANT:
To help you run Olama locally, please provide more details on the specific question. This will allow me to give a more tailored and accurate response. Here are some possible areas we could explore:

1. **Installing Dependencies**: What operating system are you using (Linux, macOS, Windows)? Have you installed all necessary dependencies such as Python, Node.js, npm?
2. **Setting Up the Environment**: How do you plan to set up your development environment? Do you need to clone the repository from GitHub?
3. **Running the Server**: After setting up the environment, what command or script are you using to start the server locally?
4. **Accessing the Application**: Once the server is running, how will you access it (e.g., through a web browser)?
5. **Troubleshooting**: If you encounter any issues during the installation or setup process, what specific error messages do you receive?

If you can provide more details on your situation, I'll be happy to help you run O

"To help you run Olama locally, please provide more details on the specific question. This will allow me to give a more tailored and accurate response. Here are some possible areas we could explore:\n\n1. **Installing Dependencies**: What operating system are you using (Linux, macOS, Windows)? Have you installed all necessary dependencies such as Python, Node.js, npm?\n2. **Setting Up the Environment**: How do you plan to set up your development environment? Do you need to clone the repository from GitHub?\n3. **Running the Server**: After setting up the environment, what command or script are you using to start the server locally?\n4. **Accessing the Application**: Once the server is running, how will you access it (e.g., through a web browser)?\n5. **Troubleshooting**: If you encounter any issues during the installation or setup process, what specific error messages do you receive?\n\nIf you can provide more details on your situation, I'll be happy to help you run Olama locally."

In [33]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
ASSISTANT:
```xml
{
  "name": "search",
  "arguments": {
    "query": "can i still join this course"
  }
}
```


'```xml\n{\n  "name": "search",\n  "arguments": {\n    "query": "can i still join this course"\n  }\n}\n```'

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

In [35]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
ASSISTANT:
```json
{
  "name": "search",
  "arguments": {
    "query": "queen gambit"
  }
}
```


'```json\n{\n  "name": "search",\n  "arguments": {\n    "query": "queen gambit"\n  }\n}\n```'

# ToyAIKit

In [37]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [38]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [39]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [40]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [41]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [44]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient # import this where your library defines it

# 1. Create an OpenAI-compatible client pointing to your local Ollama server
ollama_openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # Ollama doesn't require a key, but a placeholder is needed to bypass the credentials check
)

# 2. Set up the runner with your local Ollama client passed into OpenAIClient
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        model="qwen2.5-coder-3b", 
        client=ollama_openai_client  # Pass the local Ollama client here
    )
)

In [45]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


D:\PROJECTS\llm-zc\.venv\Lib\site-packages\toyaikit\chat\runners.py:283: UnknownModelWarning: No pricing data for model 'qwen2.5-coder-3b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [46]:
result.cost

In [47]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseOutputMessage(id='msg_330435', content=[ResponseOutputText(annotations=[], text='<tool>\n{"name": "search", "arguments": {"query": "olama run"}}\n</tool>', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]

In [ ]:
runner.run()

You: check


-> Response received
